In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Custom gates
#Amitava Chakraborty - 18th Jan 2018
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

In [ ]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from math import pi
# Importing standard Qiskit libraries and configuring account
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister, transpile
from qiskit_backend_config import execute
from qiskit.visualization import circuit_drawer
from qiskit.quantum_info import state_fidelity
from qiskit_backend_config import BasicAer

backend = BasicAer.get_backend('unitary_simulator')

In [ ]:
import numpy as np
#'Barrier' cannot be found in quantum computing textbooks because 
#it is not a standard primitive of quantum information theory like unitary gates and quantum circuits
#The barrier is a directive for circuit compilation 
#to separate pieces of a circuit so that any optimizations or re-writes 
#are constrained to only act between barriers 
#if there are no barriers they act on the whole circuit
#This only comes into play when using the transpile or execute functions in Qiskit 
#(because execute includes a transpile step).

#Example
#If a circuit has several 1-qubit gates in a row acting on the same qubit 
#these can be combined into a single 1-qubit gate. 
#If you wanted to explicitly prevent this sort of behaviour you can place a barrier between them.

#Step 1 : Create a 1-qubit circuit with several gates
qr = QuantumRegister(1)
circuit1 = QuantumCircuit(qr)
circuit1.p(0.2, qr[0])
circuit1.u(np.pi/2, 0.1, 0.2, qr[0])
circuit1.u(0.1, 0.2, 0.3, qr[0])
circuit1.draw()

In [ ]:
#If we transpile it these gates are combined using the default settings
circuit1t = transpile(circuit1)
circuit1t.draw()

In [ ]:
original_depth = circuit1.depth()

In [ ]:
import numpy as np
#Now if we wanted to stop the gates being combined we could add barriers:
qr = QuantumRegister(1)
circuit2 = QuantumCircuit(qr)
circuit2.p(0.2, qr[0])
circuit2.barrier(qr)
circuit2.u(np.pi/2, 0.1, 0.2, qr[0])
circuit2.barrier(qr)
circuit2.u(0.1, 0.2, 0.3, qr[0])
circuit2.draw()

In [ ]:
#In this case transpiling wont change the circuit:
circuit2t = transpile(circuit2)
circuit2t.draw()

In [ ]:
#Barriers do not add to depth
print (original_depth - circuit1.depth())